In [ ]:
# 🤖 AI Reliability & Agent Quality Automation
> **Purpose**: Automated agent response quality evaluation with scientific precision  
> **Capabilities**: Accuracy scoring | Hallucination detection | Grounding validation | Regression checks  
> **Value**: Makes agent tuning scientific | Enables UAT sign-off with evidence | Builds confidence in production rollout  
> **Workspace**: FabricJumpstart

---

⚠️ **IMPORTANT DISCLAIMER**: AI can make mistakes. Please verify all results and use this evaluation tool as a guide for assessing agent quality, not as the sole determination of production readiness. Always perform human review and validation before deploying AI agents to production environments.

---

## 🎯 Practical Guidance for Agent Consistency & Speed

**Before running quality tests, ensure your agent follows these proven best practices:**

### ✅ 1. **Use Verified Answers for Top Questions**
- **Biggest win for stability and performance**
- Pre-configure answers for your most frequent 10-20 questions
- Eliminates variability and ensures instant, accurate responses
- Reduces token costs and latency for common queries

### ✅ 2. **Ground Agents on Power BI Visuals**
- **Reduces query exploration paths**
- Point agents to specific reports/dashboards instead of raw tables
- Visuals already contain business logic and aggregations
- Prevents agents from re-inventing metrics incorrectly

### ✅ 3. **Narrow AI Data Schema**
- **Fewer columns = fewer candidate queries**
- Expose only business-relevant columns to the agent
- Hide technical IDs, audit timestamps, and internal metadata
- Simplifies agent reasoning and improves accuracy

### ✅ 4. **Use Explicit, Business-Approved Measures**
- **Avoids recomputation and ambiguity**
- Define DAX measures for all key metrics (Revenue, Profit, Growth %)
- Use semantic model measures instead of letting agent calculate
- Ensures consistency with existing Power BI reports

### ✅ 5. **Design for Guided Analytics**
- **FDA performs best for curated, analytical questions — not free-form chat bots**
- Focus on specific business questions with known answers
- Provide example questions to guide users
- Set clear expectations about agent capabilities and scope

---

**💡 TIP**: This notebook will help you validate that your agent meets quality standards. If tests fail, revisit the best practices above before re-tuning your agent.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 ▸ Configuration & Dependencies
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import json, datetime, warnings, re, time
from typing import Dict, List, Tuple
from IPython.display import display, HTML
import openpyxl  # For Excel file handling

warnings.filterwarnings("ignore")

# ── Workspace context ─────────────────────────────────────────────────────────
WORKSPACE_ID   = "f2adc10e-785a-4d71-91ec-ce4ee3aefef6"
WORKSPACE_NAME = "FabricJumpstart"

# ── Agent configuration ───────────────────────────────────────────────────────
AGENT_ENDPOINT = ""  # Paste your Fabric Data Agent REST API endpoint here
                     # Format: https://api.fabric.microsoft.com/v1/agents/{agent_id}/invoke

# ── Quality thresholds (adjust based on your requirements) ────────────────────
ACCURACY_THRESHOLD       = 0.80   # 80% accuracy required for UAT sign-off
HALLUCINATION_THRESHOLD  = 0.10   # Max 10% hallucination rate acceptable
GROUNDING_THRESHOLD      = 0.85   # 85% grounding success required

# ── UX & Cost thresholds ──────────────────────────────────────────────────────
LATENCY_THRESHOLD_SEC    = 10.0   # Flag responses > 10s (user abandonment risk)
COST_PER_1K_PROMPT_TOKENS    = 0.03   # Example: GPT-4 pricing $0.03/1K prompt tokens
COST_PER_1K_COMPLETION_TOKENS = 0.06   # Example: GPT-4 pricing $0.06/1K completion tokens

# ⚠️  COST DISCLAIMER: The cost estimates provided are approximate calculations based on
# token usage and pricing rates. Actual costs may vary depending on your Azure subscription,
# model deployment, region, pricing tier, and token estimation accuracy. Always verify costs
# in Azure Cost Management before production deployment.

# ── Trending & persistence ────────────────────────────────────────────────────
STORE_RESULTS_IN_DELTA   = False  # Set True to enable Delta Lake persistence
LAKEHOUSE_TABLE_PATH     = "Tables/agent_quality_results"  # Update with your path

print("✅ Configuration loaded")
print(f"   Workspace        : {WORKSPACE_NAME} ({WORKSPACE_ID})")
print(f"   Accuracy goal    : {ACCURACY_THRESHOLD*100:.0f}%")
print(f"   Hallucination max: {HALLUCINATION_THRESHOLD*100:.0f}%")
print(f"   Grounding goal   : {GROUNDING_THRESHOLD*100:.0f}%")
print(f"   Latency threshold: {LATENCY_THRESHOLD_SEC}s (UX abandonment limit)")
print(f"   Cost tracking    : Enabled (${COST_PER_1K_PROMPT_TOKENS}/1K prompt, ${COST_PER_1K_COMPLETION_TOKENS}/1K completion)")
print(f"   ⚠️  Cost estimates are approximate and may vary based on actual usage")
print(f"   Delta persistence: {'Enabled' if STORE_RESULTS_IN_DELTA else 'Disabled'}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 ▸ Load Test Cases from Excel File
#
#  Expected Excel format:
#    Column A: "question"     - The prompt/question for the agent
#    Column B: "expected_answer" - The ground truth / expected response
#  Optional columns: "category", "difficulty", "grounding_source"
# ─────────────────────────────────────────────────────────────────────────────

# ── Option 1: Upload from local file system ──────────────────────────────────
# Uncomment and set the path to your Excel file:
# EXCEL_FILE_PATH = "/lakehouse/default/Files/agent_test_cases.xlsx"

# ── Option 2: Upload via widget (interactive) ────────────────────────────────
print("📂 LOAD TEST CASES FROM EXCEL\n")
print("   Option 1: Set EXCEL_FILE_PATH variable above (recommended for automation)")
print("   Option 2: Upload file to Lakehouse Files and specify path\n")

# For this demo, we'll create a sample test dataset programmatically
# Replace this with pd.read_excel(EXCEL_FILE_PATH) in production

sample_test_cases = [
    {
        "question": "What is the total revenue for Q1 2024?",
        "expected_answer": "Total Q1 2024 revenue was $15.2M",
        "category": "Financial Metrics",
        "grounding_source": "Lakehouse: sales_summary"
    },
    {
        "question": "Which product category had the highest sales last month?",
        "expected_answer": "Electronics had the highest sales at $3.8M",
        "category": "Sales Analysis",
        "grounding_source": "Semantic Model: ProductSales"
    },
    {
        "question": "How many active customers do we have?",
        "expected_answer": "We have 12,450 active customers as of March 2024",
        "category": "Customer Metrics",
        "grounding_source": "Lakehouse: customer_table"
    },
    {
        "question": "What was the average order value in February?",
        "expected_answer": "Average order value in February was $127.50",
        "category": "Order Analytics",
        "grounding_source": "Semantic Model: OrderMetrics"
    },
    {
        "question": "Show me the top 3 sales regions by revenue",
        "expected_answer": "Top regions: 1) West ($8.2M), 2) East ($6.5M), 3) Central ($4.1M)",
        "category": "Regional Analysis",
        "grounding_source": "Lakehouse: regional_sales"
    },
]

df_test_cases = pd.DataFrame(sample_test_cases)

# ── Load from Excel (production workflow) ─────────────────────────────────────
# Uncomment this block when using your own Excel file:
#
# try:
#     df_test_cases = pd.read_excel(
#         EXCEL_FILE_PATH,
#         engine='openpyxl',
#         sheet_name=0  # First sheet
#     )
#     
#     # Validate required columns
#     required_cols = ["question", "expected_answer"]
#     missing = [c for c in required_cols if c not in df_test_cases.columns]
#     if missing:
#         raise ValueError(f"Missing required columns: {missing}")
#     
#     print(f"✅ Loaded {len(df_test_cases)} test cases from Excel")
#     
# except Exception as e:
#     print(f"❌ Failed to load Excel file: {e}")
#     print("   Using sample test cases instead")
#     df_test_cases = pd.DataFrame(sample_test_cases)

print(f"✅ Loaded {len(df_test_cases)} test cases")
print(f"   Columns: {list(df_test_cases.columns)}\n")

print("📋 Test Case Preview:")
display(HTML(df_test_cases.head().to_html(index=False, escape=False)))

# ── Initialize results dataframe ──────────────────────────────────────────────
df_test_cases["actual_answer"] = ""
df_test_cases["accuracy_score"] = 0.0
df_test_cases["hallucination_flag"] = False
df_test_cases["grounding_validated"] = False
df_test_cases["test_timestamp"] = datetime.datetime.utcnow().isoformat() + "Z"

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 ▸ Authentication & Agent Invocation Helper
# ─────────────────────────────────────────────────────────────────────────────
import requests

# ── Acquire bearer token ──────────────────────────────────────────────────────
try:
    token = notebookutils.credentials.getToken('pbi')
    HEADERS = {
        "Authorization": f"Bearer {token}",
        "Content-Type":  "application/json"
    }
    print("✅ Bearer token acquired via notebookutils (user identity)")
except Exception as e:
    print(f"⚠️  notebookutils token failed: {e}")
    print("   → For service-principal pipelines, use MSAL authentication.")
    HEADERS = {}

# ── Agent invocation function ─────────────────────────────────────────────────
def invoke_fabric_agent(question: str, headers: dict, endpoint: str) -> Dict:
    """
    Invoke a Fabric Data Agent with a question and return the response.
    Tracks latency and token usage for UX and cost analysis.
    
    Args:
        question: The user's question/prompt
        headers: Authentication headers
        endpoint: Agent API endpoint
        
    Returns:
        dict with 'answer', 'sources', 'confidence', 'error', 'latency_sec',
        'prompt_tokens', 'completion_tokens', 'total_tokens'
    """
    import random
    
    start_time = time.time()
    
    if not endpoint:
        # Simulate agent response for demo purposes
        time.sleep(random.uniform(0.5, 2.5))  # Simulate variable latency
        latency = time.time() - start_time
        
        # Simulate token usage (rough estimate based on question/answer length)
        prompt_tokens = len(question.split()) * 1.3  # Rough tokenization estimate
        completion_tokens = random.randint(50, 200)
        
        return {
            "answer": f"[SIMULATED] This would be the agent's response to: {question[:50]}...",
            "sources": ["Lakehouse: demo_table", "Semantic Model: DemoModel"],
            "confidence": 0.85,
            "error": None,
            "latency_sec": latency,
            "prompt_tokens": int(prompt_tokens),
            "completion_tokens": completion_tokens,
            "total_tokens": int(prompt_tokens + completion_tokens)
        }
    
    payload = {
        "question": question,
        "conversationId": None,  # New conversation for each test
        "includeSourceReferences": True
    }
    
    try:
        resp = requests.post(endpoint, json=payload, headers=headers, timeout=60)
        resp.raise_for_status()
        latency = time.time() - start_time
        
        data = resp.json()
        
        # Extract token usage if available (API-specific field names may vary)
        usage = data.get("usage", {})
        prompt_tokens = usage.get("prompt_tokens", usage.get("promptTokens", 0))
        completion_tokens = usage.get("completion_tokens", usage.get("completionTokens", 0))
        total_tokens = usage.get("total_tokens", usage.get("totalTokens", prompt_tokens + completion_tokens))
        
        return {
            "answer": data.get("answer", ""),
            "sources": data.get("sources", []),
            "confidence": data.get("confidence", 0.0),
            "error": None,
            "latency_sec": latency,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens
        }
        
    except requests.exceptions.Timeout:
        latency = time.time() - start_time
        return {"answer": "", "sources": [], "confidence": 0.0, "error": "Timeout",
                "latency_sec": latency, "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    except requests.exceptions.HTTPError as e:
        latency = time.time() - start_time
        return {"answer": "", "sources": [], "confidence": 0.0, 
                "error": f"HTTP {e.response.status_code}: {e.response.text[:200]}",
                "latency_sec": latency, "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    except Exception as e:
        latency = time.time() - start_time
        return {"answer": "", "sources": [], "confidence": 0.0, "error": str(e)[:200],
                "latency_sec": latency, "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

print("✅ Agent invocation helper ready")
if not AGENT_ENDPOINT:
    print("   ⚠️  AGENT_ENDPOINT not set — using simulated responses for demo")
    print("   Set AGENT_ENDPOINT in Cell 1 to test against a live Fabric Data Agent")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 ▸ Iterate Through Test Cases & Invoke Agent
# ─────────────────────────────────────────────────────────────────────────────

print("🤖 RUNNING AGENT QUALITY EVALUATION …\n")
print(f"   Testing {len(df_test_cases)} questions against agent\n")

results_list = []

for idx, row in df_test_cases.iterrows():
    question = row["question"]
    expected = row["expected_answer"]
    
    print(f"   [{idx+1}/{len(df_test_cases)}] Testing: {question[:60]}...", end=" ")
    
    # ── Invoke agent ──────────────────────────────────────────────────────────
    response = invoke_fabric_agent(question, HEADERS, AGENT_ENDPOINT)
    
    # ── Store result ──────────────────────────────────────────────────────────
    df_test_cases.at[idx, "actual_answer"] = response["answer"]
    df_test_cases.at[idx, "agent_confidence"] = response.get("confidence", 0.0)
    df_test_cases.at[idx, "agent_sources"] = json.dumps(response.get("sources", []))
    df_test_cases.at[idx, "agent_error"] = response.get("error", "")
    
    # ── Store UX & cost metrics ───────────────────────────────────────────────
    df_test_cases.at[idx, "latency_sec"] = response.get("latency_sec", 0.0)
    df_test_cases.at[idx, "prompt_tokens"] = response.get("prompt_tokens", 0)
    df_test_cases.at[idx, "completion_tokens"] = response.get("completion_tokens", 0)
    df_test_cases.at[idx, "total_tokens"] = response.get("total_tokens", 0)
    
    # Calculate cost for this query
    cost = (
        (response.get("prompt_tokens", 0) / 1000) * COST_PER_1K_PROMPT_TOKENS +
        (response.get("completion_tokens", 0) / 1000) * COST_PER_1K_COMPLETION_TOKENS
    )
    df_test_cases.at[idx, "estimated_cost_usd"] = cost
    
    if response.get("error"):
        print(f"❌ Error: {response['error'][:50]}")
    else:
        print(f"✅ ({response.get('latency_sec', 0):.1f}s)")

print(f"\n✅ Agent invocation complete\n")
print("=" * 80)
print("  AGENT RESPONSES & UX METRICS (preview)")
print("=" * 80)
display(HTML(df_test_cases[["question", "actual_answer", "latency_sec", "estimated_cost_usd"]]
             .head(10).to_html(index=False, escape=False)))

# ── UX & Cost Summary ─────────────────────────────────────────────────────────
avg_latency = df_test_cases["latency_sec"].mean()
slow_queries = df_test_cases[df_test_cases["latency_sec"] > LATENCY_THRESHOLD_SEC]
total_cost = df_test_cases["estimated_cost_usd"].sum()
avg_cost_per_query = df_test_cases["estimated_cost_usd"].mean()
total_tokens = df_test_cases["total_tokens"].sum()

print(f"\n📊 UX & COST METRICS:")
print(f"   Average latency      : {avg_latency:.2f}s")
print(f"   Slow queries (>{LATENCY_THRESHOLD_SEC}s): {len(slow_queries)} ({len(slow_queries)/len(df_test_cases)*100:.0f}%)")
print(f"   Total tokens used    : {total_tokens:,}")
print(f"   Total estimated cost : ${total_cost:.4f}")
print(f"   Avg cost per query   : ${avg_cost_per_query:.4f}")
print(f"   ⚠️  Cost estimates are approximate — verify in Azure Cost Management")

if not slow_queries.empty:
    print(f"\n⚠️  {len(slow_queries)} slow query(ies) detected (user abandonment risk):")
    display(HTML(slow_queries[["question", "latency_sec"]].to_html(index=False, escape=False)))
else:
    print(f"\n✅ All responses within acceptable latency threshold")

# ── Production cost projection ────────────────────────────────────────────────
queries_per_month = 10000  # Example: 10K queries/month
monthly_cost = avg_cost_per_query * queries_per_month

print(f"\n💰 PRODUCTION COST PROJECTION (ESTIMATE ONLY):")
print(f"   If running {queries_per_month:,} queries/month:")
print(f"   → Estimated monthly cost: ${monthly_cost:.2f}")
print(f"   → Annual cost estimate  : ${monthly_cost * 12:.2f}")
print(f"   ⚠️  These are rough estimates — actual costs may vary significantly")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 ▸ Response Comparison & Accuracy Scoring
#
#  Compares expected vs actual answers using multiple methods:
#  1. Exact match (case-insensitive)
#  2. Semantic similarity (token overlap, Jaccard)
#  3. Key entity extraction (numbers, dates, names)
# ─────────────────────────────────────────────────────────────────────────────

def extract_numbers(text: str) -> List[float]:
    """Extract all numbers from text for comparison."""
    if not text:
        return []
    # Match numbers including decimals, percentages, currency symbols
    pattern = r'[$€£¥]?\s*(\d+(?:,\d{3})*(?:\.\d+)?)\s*[%MKB]?'
    matches = re.findall(pattern, str(text))
    return [float(m.replace(',', '')) for m in matches]

def calculate_token_overlap(expected: str, actual: str) -> float:
    """Calculate Jaccard similarity between tokenized texts."""
    if not expected or not actual:
        return 0.0
    
    # Tokenize and normalize
    tokens_exp = set(re.findall(r'\w+', expected.lower()))
    tokens_act = set(re.findall(r'\w+', actual.lower()))
    
    if not tokens_exp or not tokens_act:
        return 0.0
    
    intersection = tokens_exp & tokens_act
    union = tokens_exp | tokens_act
    
    return len(intersection) / len(union) if union else 0.0

def calculate_accuracy_score(expected: str, actual: str) -> Tuple[float, str]:
    """
    Calculate accuracy score between expected and actual answers.
    Returns: (score between 0.0-1.0, explanation string)
    """
    expected = str(expected).strip()
    actual = str(actual).strip()
    
    if not actual or actual == "":
        return 0.0, "No response from agent"
    
    # ── Method 1: Exact match (case-insensitive) ──────────────────────────────
    if expected.lower() == actual.lower():
        return 1.0, "Exact match"
    
    # ── Method 2: Token overlap (semantic similarity) ─────────────────────────
    token_score = calculate_token_overlap(expected, actual)
    
    # ── Method 3: Numerical accuracy (critical for financial/metric questions)
    nums_exp = extract_numbers(expected)
    nums_act = extract_numbers(actual)
    
    numeric_match = False
    if nums_exp and nums_act:
        # Check if key numbers match (within 1% tolerance for rounding)
        numeric_match = all(
            any(abs(n_act - n_exp) / max(n_exp, 1) < 0.01 for n_act in nums_act)
            for n_exp in nums_exp
        )
    
    # ── Combined scoring ──────────────────────────────────────────────────────
    if numeric_match and token_score >= 0.5:
        score = 0.90  # High score: correct numbers + reasonable semantic match
        explanation = f"Numerical match + {token_score*100:.0f}% token overlap"
    elif numeric_match:
        score = 0.75  # Medium-high: correct numbers but poor phrasing
        explanation = f"Numerical match but low token overlap ({token_score*100:.0f}%)"
    elif token_score >= 0.7:
        score = 0.80  # Good semantic match
        explanation = f"Strong semantic similarity ({token_score*100:.0f}%)"
    elif token_score >= 0.5:
        score = 0.60  # Moderate match
        explanation = f"Moderate similarity ({token_score*100:.0f}%)"
    elif token_score >= 0.3:
        score = 0.40  # Weak match
        explanation = f"Weak similarity ({token_score*100:.0f}%)"
    else:
        score = 0.20  # Poor match
        explanation = f"Poor similarity ({token_score*100:.0f}%)"
    
    return score, explanation

# ── Apply scoring to all test cases ───────────────────────────────────────────
print("📊 CALCULATING ACCURACY SCORES …\n")

for idx, row in df_test_cases.iterrows():
    expected = row["expected_answer"]
    actual = row["actual_answer"]
    
    score, explanation = calculate_accuracy_score(expected, actual)
    
    df_test_cases.at[idx, "accuracy_score"] = score
    df_test_cases.at[idx, "score_explanation"] = explanation

print("✅ Accuracy scoring complete\n")
print("=" * 80)
print("  ACCURACY RESULTS")
print("=" * 80)

display(HTML(df_test_cases[["question", "accuracy_score", "score_explanation"]]
             .sort_values("accuracy_score", ascending=False)
             .to_html(index=False, escape=False)))

# ── Summary statistics ────────────────────────────────────────────────────────
avg_accuracy = df_test_cases["accuracy_score"].mean()
pass_count = len(df_test_cases[df_test_cases["accuracy_score"] >= ACCURACY_THRESHOLD])
fail_count = len(df_test_cases) - pass_count

print(f"\n📈 Accuracy Summary:")
print(f"   Average accuracy  : {avg_accuracy*100:.1f}%")
print(f"   Tests passed      : {pass_count}/{len(df_test_cases)} ({pass_count/len(df_test_cases)*100:.0f}%)")
print(f"   Tests failed      : {fail_count}/{len(df_test_cases)} ({fail_count/len(df_test_cases)*100:.0f}%)")
print(f"   Threshold         : {ACCURACY_THRESHOLD*100:.0f}%")

if avg_accuracy >= ACCURACY_THRESHOLD:
    print(f"\n✅ PASS — Average accuracy meets threshold for UAT sign-off")
else:
    print(f"\n🔴 FAIL — Average accuracy below threshold. Agent tuning required.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 ▸ Hallucination Detection
#
#  Flags responses that contain fabricated data or unsupported claims:
#  1. Numeric hallucinations (invented numbers not in expected answer)
#  2. Entity hallucinations (names, places, products not in source)
#  3. Confidence mismatches (high confidence but low accuracy)
# ─────────────────────────────────────────────────────────────────────────────

def detect_hallucination(expected: str, actual: str, accuracy_score: float,
                        agent_confidence: float = 0.0) -> Tuple[bool, List[str]]:
    """
    Detect if the agent's response contains hallucinated information.
    
    Returns: (is_hallucination: bool, reasons: List[str])
    """
    hallucination_flags = []
    
    # ── 1. Numerical hallucination ────────────────────────────────────────────
    # Agent provides numbers not present in expected answer
    nums_exp = set(extract_numbers(expected))
    nums_act = set(extract_numbers(actual))
    
    if nums_act and not nums_exp:
        # Agent provided numbers when none were expected
        hallucination_flags.append("Numerical hallucination: unexpected numbers in response")
    elif nums_act and nums_exp:
        unexpected_nums = nums_act - nums_exp
        # Allow for reasonable tolerance (e.g., rounding differences)
        significant_diff = any(
            not any(abs(n_act - n_exp) / max(n_exp, 1) < 0.05 for n_exp in nums_exp)
            for n_act in unexpected_nums
        )
        if significant_diff and len(unexpected_nums) > 0:
            hallucination_flags.append(f"Numerical mismatch: {list(unexpected_nums)[:3]}")
    
    # ── 2. Confidence-accuracy mismatch ───────────────────────────────────────
    # High confidence but low accuracy = likely hallucination
    if agent_confidence > 0.8 and accuracy_score < 0.5:
        hallucination_flags.append(
            f"Confidence mismatch: {agent_confidence*100:.0f}% confidence but "
            f"{accuracy_score*100:.0f}% accuracy"
        )
    
    # ── 3. Response length anomaly ────────────────────────────────────────────
    # Overly verbose responses may contain fabricated details
    if len(actual) > len(expected) * 3 and accuracy_score < 0.6:
        hallucination_flags.append("Excessive verbosity with low accuracy (possible fabrication)")
    
    # ── 4. Pattern-based detection ────────────────────────────────────────────
    # Common hallucination phrases
    hallucination_phrases = [
        r"I don[''']t have.*information",
        r"I cannot.*find",
        r"based on my knowledge.*may not be accurate",
        r"I[''']m not sure",
        r"This is just an estimate"
    ]
    for pattern in hallucination_phrases:
        if re.search(pattern, actual, re.IGNORECASE):
            if accuracy_score < 0.5:  # Only flag if answer also inaccurate
                hallucination_flags.append("Uncertain phrasing with low accuracy")
                break
    
    is_hallucination = len(hallucination_flags) > 0
    return is_hallucination, hallucination_flags

# ── Apply hallucination detection ─────────────────────────────────────────────
print("🔍 DETECTING HALLUCINATIONS …\n")

for idx, row in df_test_cases.iterrows():
    expected = row["expected_answer"]
    actual = row["actual_answer"]
    accuracy = row["accuracy_score"]
    confidence = row.get("agent_confidence", 0.0)
    
    is_hallucination, reasons = detect_hallucination(expected, actual, accuracy, confidence)
    
    df_test_cases.at[idx, "hallucination_flag"] = is_hallucination
    df_test_cases.at[idx, "hallucination_reasons"] = "; ".join(reasons) if reasons else ""

print("✅ Hallucination detection complete\n")

# ── Show flagged cases ─────────────────────────────────────────────────────────
hallucinations = df_test_cases[df_test_cases["hallucination_flag"] == True]

if not hallucinations.empty:
    print("=" * 80)
    print(f"  ⚠️  {len(hallucinations)} HALLUCINATIONS DETECTED")
    print("=" * 80)
    display(HTML(hallucinations[["question", "actual_answer", "accuracy_score", "hallucination_reasons"]]
                 .to_html(index=False, escape=False)))
else:
    print("✅ No hallucinations detected\n")

# ── Summary ────────────────────────────────────────────────────────────────────
hallucination_rate = len(hallucinations) / len(df_test_cases)

print(f"\n📊 Hallucination Summary:")
print(f"   Hallucination rate: {hallucination_rate*100:.1f}%")
print(f"   Threshold         : {HALLUCINATION_THRESHOLD*100:.0f}%")

if hallucination_rate <= HALLUCINATION_THRESHOLD:
    print(f"\n✅ PASS — Hallucination rate within acceptable range")
else:
    print(f"\n🔴 FAIL — Hallucination rate exceeds threshold. Review agent grounding.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 ▸ Grounding Validation (Semantic Model vs Lakehouse)
#
#  Validates that agent responses are properly grounded in configured data sources:
#  - Checks if cited sources match expected grounding source
#  - Flags responses with no source citations
#  - Detects cross-source contamination (e.g., Lakehouse data attributed to semantic model)
# ─────────────────────────────────────────────────────────────────────────────

def validate_grounding(agent_sources: str, expected_source: str) -> Tuple[bool, str]:
    """
    Validate that agent's cited sources match expected grounding source.
    
    Args:
        agent_sources: JSON string of sources returned by agent
        expected_source: Expected grounding source (e.g., "Lakehouse: sales_table")
        
    Returns: (is_valid: bool, explanation: str)
    """
    if not expected_source:
        return True, "No expected source specified"
    
    try:
        sources_list = json.loads(agent_sources) if agent_sources else []
    except:
        sources_list = []
    
    if not sources_list:
        return False, "No sources cited by agent"
    
    # ── Parse expected source type and name ───────────────────────────────────
    expected_lower = expected_source.lower()
    is_lakehouse_expected = "lakehouse" in expected_lower
    is_semantic_model_expected = "semantic" in expected_lower or "model" in expected_lower
    
    # ── Check agent sources ────────────────────────────────────────────────────
    agent_sources_str = " ".join([str(s).lower() for s in sources_list])
    
    has_lakehouse = "lakehouse" in agent_sources_str or "lake" in agent_sources_str
    has_semantic_model = "semantic" in agent_sources_str or "model" in agent_sources_str or "dataset" in agent_sources_str
    
    # ── Validation logic ──────────────────────────────────────────────────────
    if is_lakehouse_expected and has_lakehouse:
        return True, "✅ Correctly grounded in Lakehouse"
    elif is_semantic_model_expected and has_semantic_model:
        return True, "✅ Correctly grounded in Semantic Model"
    elif is_lakehouse_expected and has_semantic_model:
        return False, "🔴 Cross-source contamination: Expected Lakehouse, got Semantic Model"
    elif is_semantic_model_expected and has_lakehouse:
        return False, "🔴 Cross-source contamination: Expected Semantic Model, got Lakehouse"
    elif not has_lakehouse and not has_semantic_model:
        return False, f"🟡 Unknown source type: {sources_list[:2]}"
    else:
        return False, f"🟡 Source mismatch: expected {expected_source}, got {sources_list[:2]}"

# ── Apply grounding validation ────────────────────────────────────────────────
print("🔗 VALIDATING GROUNDING SOURCES …\n")

for idx, row in df_test_cases.iterrows():
    agent_sources = row.get("agent_sources", "[]")
    expected_source = row.get("grounding_source", "")
    
    is_valid, explanation = validate_grounding(agent_sources, expected_source)
    
    df_test_cases.at[idx, "grounding_validated"] = is_valid
    df_test_cases.at[idx, "grounding_explanation"] = explanation

print("✅ Grounding validation complete\n")

# ── Show grounding failures ───────────────────────────────────────────────────
grounding_failures = df_test_cases[df_test_cases["grounding_validated"] == False]

if not grounding_failures.empty:
    print("=" * 80)
    print(f"  ⚠️  {len(grounding_failures)} GROUNDING FAILURES DETECTED")
    print("=" * 80)
    display(HTML(grounding_failures[["question", "grounding_source", "agent_sources", "grounding_explanation"]]
                 .to_html(index=False, escape=False)))
else:
    print("✅ All responses properly grounded\n")

# ── Summary ────────────────────────────────────────────────────────────────────
grounding_success_rate = len(df_test_cases[df_test_cases["grounding_validated"] == True]) / len(df_test_cases)

print(f"\n📊 Grounding Summary:")
print(f"   Grounding success rate: {grounding_success_rate*100:.1f}%")
print(f"   Threshold             : {GROUNDING_THRESHOLD*100:.0f}%")

if grounding_success_rate >= GROUNDING_THRESHOLD:
    print(f"\n✅ PASS — Grounding validation meets threshold")
else:
    print(f"\n🔴 FAIL — Grounding failures exceed threshold. Review agent data sources.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FINAL CELL ▸ Executive Summary & Delta Lake Persistence (Trending Over Time)
# ─────────────────────────────────────────────────────────────────────────────

# ══════════════════════════════════════════════════════════════════════════════
# EXECUTIVE SUMMARY — UAT SIGN-OFF REPORT
# ══════════════════════════════════════════════════════════════════════════════

test_run_summary = {
    "run_timestamp_utc":        datetime.datetime.utcnow().isoformat() + "Z",
    "workspace_name":           WORKSPACE_NAME,
    "total_test_cases":         len(df_test_cases),
    "avg_accuracy_pct":         df_test_cases["accuracy_score"].mean() * 100,
    "tests_passed":             len(df_test_cases[df_test_cases["accuracy_score"] >= ACCURACY_THRESHOLD]),
    "tests_failed":             len(df_test_cases[df_test_cases["accuracy_score"] < ACCURACY_THRESHOLD]),
    "hallucinations_detected":  len(df_test_cases[df_test_cases["hallucination_flag"] == True]),
    "hallucination_rate_pct":   (len(df_test_cases[df_test_cases["hallucination_flag"] == True]) / len(df_test_cases)) * 100,
    "grounding_success_pct":    (len(df_test_cases[df_test_cases["grounding_validated"] == True]) / len(df_test_cases)) * 100,
    "grounding_failures":       len(df_test_cases[df_test_cases["grounding_validated"] == False]),
    "agent_endpoint":           AGENT_ENDPOINT if AGENT_ENDPOINT else "SIMULATED",
    # UX & Cost metrics
    "avg_latency_sec":          df_test_cases["latency_sec"].mean(),
    "slow_queries_count":       len(df_test_cases[df_test_cases["latency_sec"] > LATENCY_THRESHOLD_SEC]),
    "slow_queries_pct":         (len(df_test_cases[df_test_cases["latency_sec"] > LATENCY_THRESHOLD_SEC]) / len(df_test_cases)) * 100,
    "total_tokens":             int(df_test_cases["total_tokens"].sum()),
    "avg_tokens_per_query":     df_test_cases["total_tokens"].mean(),
    "total_cost_usd":           df_test_cases["estimated_cost_usd"].sum(),
    "avg_cost_per_query_usd":   df_test_cases["estimated_cost_usd"].mean(),
}

# ── Overall pass/fail determination ───────────────────────────────────────────
overall_pass = (
    test_run_summary["avg_accuracy_pct"] >= ACCURACY_THRESHOLD * 100 and
    test_run_summary["hallucination_rate_pct"] <= HALLUCINATION_THRESHOLD * 100 and
    test_run_summary["grounding_success_pct"] >= GROUNDING_THRESHOLD * 100 and
    test_run_summary["slow_queries_pct"] < 20  # < 20% of queries should be slow
)

test_run_summary["overall_pass"] = overall_pass
test_run_summary["uat_sign_off_ready"] = "✅ YES" if overall_pass else "🔴 NO"

print("=" * 80)
print("  🤖 AGENT QUALITY EVALUATION — EXECUTIVE SUMMARY")
print("=" * 80)
for k, v in test_run_summary.items():
    if isinstance(v, float):
        print(f"  {k:<30} {v:.2f}")
    else:
        print(f"  {k:<30} {v}")
print("=" * 80)

# ── Detailed results table ────────────────────────────────────────────────────
print("\n📋 DETAILED TEST RESULTS (all cases):")
display(HTML(df_test_cases[[
    "question", "expected_answer", "actual_answer", "accuracy_score",
    "latency_sec", "estimated_cost_usd", "hallucination_flag", 
    "grounding_validated", "score_explanation"
]].to_html(index=False, escape=False)))

# ══════════════════════════════════════════════════════════════════════════════
# REGRESSION TESTING — Compare with Previous Run
# ══════════════════════════════════════════════════════════════════════════════

print("\n🔄 REGRESSION CHECK:")
print("   To enable regression testing:")
print("   1. Enable Delta Lake persistence (set STORE_RESULTS_IN_DELTA = True)")
print("   2. Re-run this notebook after agent republish")
print("   3. Compare current run vs previous run from Delta table\n")

# ══════════════════════════════════════════════════════════════════════════════
# DELTA LAKE PERSISTENCE — Store Results for Trending Over Time
# ══════════════════════════════════════════════════════════════════════════════

if STORE_RESULTS_IN_DELTA:
    print("💾 PERSISTING RESULTS TO DELTA LAKE …\n")
    
    try:
        # ── Prepare dataframes for persistence ────────────────────────────────
        # Table 1: Test run summary (one row per run)
        df_summary = pd.DataFrame([test_run_summary])
        
        # Table 2: Detailed test results (one row per test case)
        df_details = df_test_cases.copy()
        df_details["run_id"] = test_run_summary["run_timestamp_utc"]
        
        # ── Write to Delta tables ─────────────────────────────────────────────
        spark.createDataFrame(df_summary).write.format("delta").mode("append") \
            .save(f"{LAKEHOUSE_TABLE_PATH}_summary")
        
        spark.createDataFrame(df_details).write.format("delta").mode("append") \
            .save(f"{LAKEHOUSE_TABLE_PATH}_details")
        
        print("   ✅ Test run summary written to Delta table: {}_summary".format(LAKEHOUSE_TABLE_PATH))
        print("   ✅ Detailed results written to Delta table: {}_details".format(LAKEHOUSE_TABLE_PATH))
        print("\n   📊 Use Power BI Direct Lake to create trending reports:")
        print("      • Accuracy trends over time (line chart)")
        print("      • Hallucination rate by test run (bar chart)")
        print("      • Grounding success rate (gauge)")
        print("      • Average latency trends (line chart)")
        print("      • Cost per query trends (line chart)")
        print("      • Regression detection (current vs previous)")
        
    except Exception as e:
        print(f"   ❌ Failed to write to Delta Lake: {e}")
        print("   Make sure a Lakehouse is attached to this notebook")
else:
    print("💡 Delta Lake persistence is DISABLED")
    print("   Set STORE_RESULTS_IN_DELTA = True in Cell 1 to enable trending over time\n")

# ══════════════════════════════════════════════════════════════════════════════
# NEXT STEPS — PRODUCTION READINESS CHECKLIST
# ══════════════════════════════════════════════════════════════════════════════

print("\n📌 NEXT STEPS FOR PRODUCTION AGENT DEPLOYMENT:\n")

if overall_pass:
    print("  ✅ AGENT READY FOR UAT SIGN-OFF\n")
    print("  1️⃣  Share this executive summary with stakeholders")
    print("  2️⃣  Document test cases in version control (Excel → Git)")
    print("  3️⃣  Enable Delta persistence for ongoing regression testing")
    print("  4️⃣  Schedule this notebook to run after each agent republish")
    print("  5️⃣  Create Power BI dashboard for quality + UX + cost trends")
    print("  6️⃣  Proceed to production rollout with confidence\n")
else:
    print("  🔴 AGENT NOT READY — TUNING REQUIRED\n")
    print("  ⭐ REVIEW BEST PRACTICES at the top of this notebook before making changes!\n")
    
    if test_run_summary["avg_accuracy_pct"] < ACCURACY_THRESHOLD * 100:
        print("  ⚠️  ACCURACY ISSUE — Apply these best practices:")
        print("     ✅ Use Verified Answers for top questions (biggest win!)")
        print("     ✅ Use explicit, business-approved measures (avoid recomputation)")
        print("     ✅ Narrow AI data schema (fewer columns = fewer errors)")
        print("     • Also: Review failed test cases above")
        print("     • Improve agent prompting/instructions")
        print("     • Add more examples to agent training data\n")
    
    if test_run_summary["hallucination_rate_pct"] > HALLUCINATION_THRESHOLD * 100:
        print("  ⚠️  HALLUCINATION ISSUE — Apply these best practices:")
        print("     ✅ Ground agents on Power BI visuals (reduces query exploration)")
        print("     ✅ Use explicit measures (prevents fabricated calculations)")
        print("     • Also: Strengthen grounding constraints")
        print("     • Add source citation requirements")
        print("     • Review agent temperature/top_p settings\n")
    
    if test_run_summary["grounding_success_pct"] < GROUNDING_THRESHOLD * 100:
        print("  ⚠️  GROUNDING ISSUE — Apply these best practices:")
        print("     ✅ Ground agents on Power BI visuals (clearer data lineage)")
        print("     ✅ Narrow AI data schema (reduces confusion between sources)")
        print("     • Also: Verify semantic model / lakehouse connections")
        print("     • Check agent data source configuration")
        print("     • Review cross-source contamination cases\n")
    
    if test_run_summary["slow_queries_pct"] >= 20:
        print("  ⚠️  UX/LATENCY ISSUE — Apply these best practices:")
        print("     ✅ Use Verified Answers for top questions (instant responses!)")
        print("     ✅ Ground on Power BI visuals (pre-aggregated data = faster)")
        print("     ✅ Narrow AI data schema (fewer columns = faster query generation)")
        print(f"     • {test_run_summary['slow_queries_count']} queries exceed {LATENCY_THRESHOLD_SEC}s threshold")
        print("     • Risk of user abandonment (users drop off after 10s)")
        print("     • Consider caching frequent queries\n")
    
    print("  ⭐ REMEMBER: Design for guided analytics, not free-form chat bots!")
    print("     FDA performs best with curated, analytical questions\n")
    print("  🔄 After applying best practices and tuning, re-run this notebook\n")

# ── Production cost estimate ──────────────────────────────────────────────────
queries_per_month = 10000  # Adjust based on expected production volume
monthly_cost = test_run_summary["avg_cost_per_query_usd"] * queries_per_month

print("💰 PRODUCTION BUDGET GUIDANCE:")
print(f"   Average cost per query  : ${test_run_summary['avg_cost_per_query_usd']:.4f}")
print(f"   If running {queries_per_month:,} queries/month:")
print(f"   → Estimated monthly cost: ${monthly_cost:.2f}")
print(f"   → Annual budget required: ${monthly_cost * 12:,.2f}")
print("   ⚠️  Apply best practices above to reduce costs (especially Verified Answers!)")

print("\n=" * 80)
print("  🎯 AGENT QUALITY EVALUATION COMPLETE")
print("=" * 80)